# T2 D08 Value Semantics - end-to-end experiment

This notebook supports either a synthetic fixture or an Analysis Artifact Repository snapshot. It selects columns by production role, compares matching with and without optional dictionary evidence, uses deterministic exact matching first, optionally calls the configured LLM for unresolved columns, and then executes KB rules deterministically at cell or declared group grain. Raw input cell values are never sent to the LLM.

## Execution stages

1. Load a fixture or a repository snapshot ID and table.
2. Select columns by production role and, optionally, explicit column name.
3. Compare deterministic retrieval with dictionary evidence included and withheld.
4. Accept unique exact role matches; send only unresolved column metadata to the LLM when enabled.
5. Review the resolved bindings. Unresolved columns remain unbound; they are not guessed.
6. Route all applicable KB entries and expose missing prerequisites as `UNSCOPED`.
7. Evaluate cell/group rules, resolve tag precedence, summarize, and export evidence.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 60)
pd.set_option('display.max_colwidth', 120)
cwd = Path.cwd().resolve()
EXPERIMENT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / 'kb' / 'value_semantics_kb_v0_2.yaml').is_file()), None)
if EXPERIMENT_ROOT is None:
    raise RuntimeError('Run this notebook from the t2_d08_Value_semantics folder or one of its children.')
if str(EXPERIMENT_ROOT) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_ROOT))

# Input selection
INPUT_SOURCE = 'fixture'       # 'fixture' or 'snapshot'
FIXTURE = 'pd'                 # pd, lgd, or ead
SNAPSHOT_ID = 'item_e243fa4e72f4'  # used only for snapshot input
TABLE_NAME = 'mr_pd_sample'         # used only for snapshot input
ELIGIBLE_ROLES = None          # e.g. {'feature', 'target', 'score'}; None means all non-technical columns
SELECTED_COLUMNS = None        # e.g. {'CURR_DPD', 'RPT_QTR'}; applied after role selection
DICTIONARY_PATH = None         # optional external .csv/.yaml dictionary to overlay by column name

# Matching controls
USE_AAR_SOURCED_DICTIONARY = True  # snapshot only: use saved AAR descriptions and confirmed sentinels
USE_DICTIONARY_FOR_MATCHING = True # False matches on name, type, and production role only
COMPARE_DICTIONARY_MODES = True
USE_LLM_FALLBACK = True       # opt-in external call for unresolved column metadata only
USE_REVIEWED_FIXTURE_BINDINGS = True  # fixture-only isolation of rule execution
BINDING_OVERRIDES = {}         # e.g. {'BAL_AMT': ['current_balance']} after human review

# Snapshot rule declarations are dataset-specific and must be supplied explicitly.
SNAPSHOT_RUNTIME_DECLARATIONS = {}
OUTPUT_ROOT = EXPERIMENT_ROOT / 'output' / 'cell_execution_v0_1'
CHECKPOINT_PATH = EXPERIMENT_ROOT / '.runtime' / 'notebook_role_adjudication_v0_2.jsonl'
D11_ENV_FILE = EXPERIMENT_ROOT.parent / 't2_d11_dir_consistency' / '.env'
print('Experiment:', EXPERIMENT_ROOT)
print('Source:', INPUT_SOURCE, '| AAR dictionary:', USE_AAR_SOURCED_DICTIONARY, '| matching dictionary:', USE_DICTIONARY_FOR_MATCHING, '| LLM fallback:', USE_LLM_FALLBACK)

Experiment: C:\Src\DataWorkbench\experiments\test-lab\t2_d08_Value_semantics
Source: fixture | AAR dictionary: True | matching dictionary: True | LLM fallback: True


In [2]:
from functions.cell_rule_execution import execute_value_semantics
from functions.column_binding_workflow import resolve_column_bindings
from functions.dictionary_io import load_dictionary, overlay_dictionary
from functions.kb_loader import load_terminology, load_value_semantics_kb
from functions.result_summary import summarize_value_semantics
from functions.role_matching import match_variable_to_roles, prepare_role_matcher
from functions.run_cell_diagnostic import export_execution, load_fixture
from functions.snapshot_input import dictionary_evidence_view, load_snapshot_input
kb = load_value_semantics_kb(EXPERIMENT_ROOT / 'kb' / 'value_semantics_kb_v0_2.yaml')
terminology = load_terminology(EXPERIMENT_ROOT / 'kb' / 'credit_risk_abbreviations_v0_3.yaml')
print(f"KB {kb['metadata']['version']}: {len(kb['semantic_roles'])} roles, {len(kb['rules'])} rules")

KB 0.2: 36 roles, 3 rules


## 1. Load and scope input

Role and explicit-column filters determine which columns enter semantic matching. The full table and full dictionary remain available to deterministic rules. Selecting too narrow a role set can intentionally produce `UNSCOPED` routes when prerequisite roles such as entity, period, segment, or event indicators are absent.

In [3]:
if INPUT_SOURCE == 'fixture':
    fixture = load_fixture(FIXTURE)
    data = fixture['data'].copy()
    full_dictionary = fixture['dictionary'].copy()
    row_reference_column = 'ROW_ID'
    declarations = fixture['declarations']
    source_label = FIXTURE.lower()
else:
    snapshot = load_snapshot_input(
        snapshot_id=SNAPSHOT_ID, table=TABLE_NAME,
        include_dictionary_metadata=USE_AAR_SOURCED_DICTIONARY,
    )
    data = snapshot['data']
    aar_sourced_dictionary = snapshot['aar_sourced_dictionary']
    full_dictionary = snapshot['dictionary']
    row_reference_column = snapshot['row_reference_column']
    declarations = SNAPSHOT_RUNTIME_DECLARATIONS
    source_label = f"snapshot_{SNAPSHOT_ID}_{TABLE_NAME}"
if DICTIONARY_PATH:
    full_dictionary = overlay_dictionary(full_dictionary, load_dictionary(DICTIONARY_PATH))
selected_dictionary = full_dictionary.loc[~full_dictionary['column_name'].eq(row_reference_column)].copy()
if ELIGIBLE_ROLES:
    selected_dictionary = selected_dictionary.loc[selected_dictionary['role'].str.casefold().isin({r.casefold() for r in ELIGIBLE_ROLES})]
if SELECTED_COLUMNS:
    missing = set(SELECTED_COLUMNS) - set(full_dictionary['column_name'])
    if missing:
        raise KeyError(f'Selected columns absent from input table: {sorted(missing)}')
    selected_dictionary = selected_dictionary.loc[selected_dictionary['column_name'].isin(SELECTED_COLUMNS)]
selected_dictionary = selected_dictionary.reset_index(drop=True)
display(Markdown(f'**Loaded:** {len(data):,} rows x {len(data.columns)} columns; **selected for matching:** {len(selected_dictionary)} columns'))
display(data.head())
if INPUT_SOURCE == 'snapshot':
    aar_view = aar_sourced_dictionary.loc[~aar_sourced_dictionary['column_name'].eq(row_reference_column)].copy()
    aar_view['description_available'] = aar_view['description'].astype(str).str.strip().ne('')
    display(Markdown(f"**AAR-sourced dictionary:** {len(aar_view)} columns; descriptions available for {int(aar_view['description_available'].sum())}. Used in matching: **{USE_AAR_SOURCED_DICTIONARY}**"))
    display(aar_view[['column_name', 'data_type', 'role', 'description', 'sentinel_value', 'description_available']])
display(selected_dictionary[['column_name', 'business_name', 'description', 'data_type', 'role', 'sentinel_value']])
display(pd.DataFrame({'declaration': declarations.keys(), 'value': [str(v) for v in declarations.values()]}))

**Loaded:** 1,200 rows x 19 columns; **selected for matching:** 18 columns

,ROW_ID,FACILITY_ID,RPT_QTR,PORTF_SEG,CURR_DPD,INT_RT,RATE_TYPE,BRR,NOI,OCC_PCT,DFLT_12M,DEFAULT_FLAG,SPREAD_BPS,MAT_DT,MAT_BALLOON_IND,PMT_TYPE,PRIN_PAYDOWN_AMT,ORIG_LTV,BAL_AMT
0,PD001_01,F0001,2022-Q1,NORTH,0,3.53,FLOATING,G2,50130.0,0.690,0.0,0,176.0,2030-12-31,0,AMORTISING,1000,0.7,250000
1,PD001_02,F0001,2022-Q2,NORTH,0,3.61,FLOATING,G2,50370.0,0.691,0.0,0,176.0,2030-12-31,0,AMORTISING,1000,0.7,249000
2,PD001_03,F0001,2022-Q3,NORTH,0,3.69,FLOATING,G2,50610.0,0.692,0.0,0,176.0,2030-12-31,0,AMORTISING,1000,0.7,248000
3,PD001_04,F0001,2022-Q4,NORTH,0,3.77,FLOATING,G2,50850.0,0.693,0.0,0,176.0,2030-12-31,0,AMORTISING,1000,0.7,247000
4,PD001_05,F0001,2023-Q1,NORTH,0,4.25,FLOATING,G2,51090.0,0.694,0.0,0,176.0,2030-12-31,0,AMORTISING,1000,0.7,246000


,column_name,business_name,description,data_type,role,sentinel_value
0,FACILITY_ID,Facility identifier,Unique facility identifier,string,identifier,
1,RPT_QTR,Reporting quarter,Ordered quarterly reporting period,string,period,
2,PORTF_SEG,Portfolio segment,Portfolio monitoring segment,string,group,
3,CURR_DPD,Current DPD,Current contractual days past due,integer,feature,
4,INT_RT,Interest rate,Current contractual interest rate,float,feature,
5,RATE_TYPE,Rate type,Fixed or floating benchmark-linked pricing basis,string,feature,
6,BRR,Borrower risk rating,Internal borrower risk rating currently in force,string,score,
7,NOI,Net operating income,Periodic net operating income,float,feature,-999
8,OCC_PCT,Occupancy percentage,Current property occupancy percentage,float,feature,
9,DFLT_12M,12 month default,Forward default outcome over the next twelve months,integer,target,


,declaration,value
0,as_of_date,2024-12-31
1,forward_horizon_by_label,{'DFLT_12M': '4 quarters'}
2,panel_end_period,2024-Q4
3,declared_exit_events,['PAID_OFF']
4,panel_scope_exclusions,['default observations']
5,field_specific_sentinel_definitions,{'NOI': -999.0}
6,variance_window,3
7,minimum_row_count,10
8,not_applicable_share_threshold,0.8


## 2. Compare matching with and without dictionary evidence

Both modes retain physical name, data type, and production role. The without-dictionary mode withholds optional business name, description, allowed values, and confirmed sentinel metadata. This is an evidence-ablation comparison; it does not alter source data.

In [4]:
matcher = prepare_role_matcher(kb, terminology)
def retrieval_view(dictionary_frame, label):
    rows = []
    for column in dictionary_frame.to_dict(orient='records'):
        match = match_variable_to_roles(column, matcher)
        rows.append({
            'column_name': column['column_name'], 'mode': label,
            'expanded_name': match['expanded_name'], 'match_status': match['match_status'],
            'exact_role': match['exact_match']['role'] if match['exact_match'] else '',
            'detailed_candidate_count': len(match['detailed_candidates']),
            'top_candidates': ';'.join(item['role'] for item in match['detailed_candidates'][:5]),
        })
    return pd.DataFrame(rows)
with_dictionary = dictionary_evidence_view(selected_dictionary, include_dictionary_metadata=True)
without_dictionary = dictionary_evidence_view(selected_dictionary, include_dictionary_metadata=False)
retrieval_with = retrieval_view(with_dictionary, 'with_dictionary')
retrieval_without = retrieval_view(without_dictionary, 'without_dictionary')
if COMPARE_DICTIONARY_MODES:
    comparison = retrieval_with.merge(retrieval_without, on='column_name', suffixes=('_with', '_without'))
    comparison['candidate_set_changed'] = comparison['top_candidates_with'] != comparison['top_candidates_without']
    display(comparison)
    display(comparison['candidate_set_changed'].value_counts().rename_axis('candidate_set_changed').reset_index(name='columns'))
else:
    display(retrieval_with if USE_DICTIONARY_FOR_MATCHING else retrieval_without)

,column_name,mode_with,expanded_name_with,match_status_with,exact_role_with,detailed_candidate_count_with,top_candidates_with,mode_without,expanded_name_without,match_status_without,exact_role_without,detailed_candidate_count_without,top_candidates_without,candidate_set_changed
0,FACILITY_ID,with_dictionary,facility id,exact_match,entity_id,0,,without_dictionary,facility id,exact_match,entity_id,0,,False
1,RPT_QTR,with_dictionary,rpt quarter,candidate_match,,12,period;default_date;forward_labels;income_measure;internal_grade,without_dictionary,rpt quarter,candidate_match,,12,period;default_date;forward_labels;income_measure;internal_grade,False
2,PORTF_SEG,with_dictionary,portf seg,candidate_match,,12,segment;period;regime_stamp;priced_rate;internal_grade,without_dictionary,portf seg,no_lexical_evidence,,12,maturity_date;priced_rate;spread_over_benchmark;segment;internal_grade,True
3,CURR_DPD,with_dictionary,current days past due,exact_match,arrears_measure,0,,without_dictionary,current days past due,exact_match,arrears_measure,0,,False
4,INT_RT,with_dictionary,interest rt,candidate_match,,12,priced_rate;credit_limit;arrears_measure;amortisation_basis;spread_over_benchmark,without_dictionary,interest rt,no_lexical_evidence,,12,amortisation_basis;priced_rate;rate_basis;spread_over_benchmark;maturity_date,True
5,RATE_TYPE,with_dictionary,rate type,exact_match,rate_basis,0,,without_dictionary,rate type,exact_match,rate_basis,0,,False
6,BRR,with_dictionary,borrower risk rating,exact_match,internal_grade,0,,without_dictionary,borrower risk rating,exact_match,internal_grade,0,,False
7,NOI,with_dictionary,net operating income,exact_match,income_measure,0,,without_dictionary,net operating income,exact_match,income_measure,0,,False
8,OCC_PCT,with_dictionary,occupancy percentage,candidate_match,,12,priced_rate;utilisation_measure;arrears_measure;internal_grade;credit_limit,without_dictionary,occupancy percentage,candidate_match,,12,utilisation_measure;priced_rate;maturity_date;default_date;income_measure,True
9,DFLT_12M,with_dictionary,dflt 12 months,candidate_match,,12,forward_labels;default_date;default_event;arrears_measure;outcome_state,without_dictionary,dflt 12 months,candidate_match,,12,forward_labels;arrears_measure;priced_rate;maturity_date;default_date,True


,candidate_set_changed,columns
0,True,11
1,False,7


## 3. Resolve semantic-role bindings

The LLM call is required only where deterministic matching cannot establish a unique exact role. Set `USE_LLM_FALLBACK=True` to use the d11 `.env` configuration. That opt-in sends selected column name/type/production-role metadata plus optional dictionary text and KB role metadata to the configured OpenAI/Azure endpoint; it never sends table cell values. Responses are validated against the structured contract and checkpointed.

In [5]:
adjudicator = None
if USE_LLM_FALLBACK:
    from functions.llm_role_adjudication import OpenAIRoleAdjudicator, configured_model, create_openai_client, load_adjudication_prompt
    prompt = load_adjudication_prompt(EXPERIMENT_ROOT / 'prompts' / 'value_semantics_role_adjudication_v0_2.txt')
    adjudicator = OpenAIRoleAdjudicator(
        client=create_openai_client(D11_ENV_FILE), prompt=prompt, model=configured_model(D11_ENV_FILE),
    )
matching_dictionary = with_dictionary if USE_DICTIONARY_FOR_MATCHING else without_dictionary
binding_results, inferred_bindings = resolve_column_bindings(
    matching_dictionary.to_dict(orient='records'), matcher, adjudicator=adjudicator,
    checkpoint_path=CHECKPOINT_PATH, binding_overrides=BINDING_OVERRIDES,
)
if INPUT_SOURCE == 'fixture' and USE_REVIEWED_FIXTURE_BINDINGS:
    selected_names = set(matching_dictionary['column_name'])
    execution_bindings = {name: roles for name, roles in fixture['bindings'].items() if name in selected_names}
    binding_authority = 'reviewed_fixture_bindings'
else:
    execution_bindings = inferred_bindings
    binding_authority = 'exact_plus_llm_and_overrides'
display(binding_results)
display(binding_results['decision'].value_counts().rename_axis('decision').reset_index(name='columns'))
print('Binding authority:', binding_authority)
print('Execution-ready bindings:', len(execution_bindings), 'of', len(matching_dictionary))
pending = binding_results.loc[~binding_results['column_name'].isin(execution_bindings)]
if not pending.empty:
    display(Markdown('**Review required:** selected columns below remain unbound and cannot trigger rules.'))
    display(pending[['column_name', 'decision', 'reason', 'error']])

,column_name,production_role,description_supplied,match_status,detailed_candidate_count,adjudication_used,checkpoint_hit,decision,direct_roles,resolved_roles,review_required,reason,response_id,error
0,FACILITY_ID,identifier,True,exact_match,0,False,False,MATCH,entity_id,entity_id,False,Deterministic exact match,,
1,RPT_QTR,period,True,candidate_match,12,True,False,MATCH,period,period,True,Business name and description indicate an ordered quarterly reporting period; detailed candidate 'period' explicitly...,resp_0e77e1dfb18b4cd5016a9db703c9e88194b247a1ae10d7e2e4,
2,PORTF_SEG,group,True,candidate_match,12,True,False,MATCH,segment,segment,True,Business name and description indicate a portfolio partition used for monitoring; allowed values are region-like cat...,resp_094dbffb52885b63016a9db705ee50819099420a715042ab6b,
3,CURR_DPD,feature,True,exact_match,0,False,False,MATCH,arrears_measure,arrears_measure,False,Deterministic exact match,,
4,INT_RT,feature,True,candidate_match,12,True,False,MATCH,priced_rate,priced_rate,True,"The variable is explicitly described as the current contractual interest rate, matching the detailed candidate price...",resp_01f8001a686581bc016a9db707320881948d136f6733c35c24,
5,RATE_TYPE,feature,True,exact_match,0,False,False,MATCH,rate_basis,rate_basis,False,Deterministic exact match,,
6,BRR,score,True,exact_match,0,False,False,MATCH,internal_grade,internal_grade,False,Deterministic exact match,,
7,NOI,feature,True,exact_match,0,False,False,MATCH,income_measure,income_measure,False,Deterministic exact match,,
8,OCC_PCT,feature,True,candidate_match,12,True,False,MATCH,utilisation_measure,utilisation_measure,True,Description and business name indicate current property occupancy percentage; the catalog explicitly maps occupancy/...,resp_06531825e5a8f091016a9db70852488196b5738f94bfa80a66,
9,DFLT_12M,target,True,candidate_match,12,True,False,MATCH,forward_labels,forward_labels,True,"The variable is a binary target described as the forward default outcome over the next twelve months, which directly...",resp_08fb8363ae91686a016a9db709c77c819580d14388e0586063,


,decision,columns
0,MATCH,16
1,MULTI_ROLE_MATCH,1
2,ERROR,1


Binding authority: reviewed_fixture_bindings
Execution-ready bindings: 18 of 18


## 4. Execute KB rules at declared grain

From this point onward execution is deterministic. The LLM does not inspect cells, decide flags, or execute rules. Missing bindings or runtime declarations appear in the plan as `UNSCOPED`.

In [6]:
execution = execute_value_semantics(
    data, full_dictionary, execution_bindings, declarations, kb, row_reference_column=row_reference_column,
)
summaries = summarize_value_semantics(data, full_dictionary, execution_bindings, kb, execution)
display(execution.execution_plan)
if not execution.execution_plan.empty:
    display(execution.execution_plan['routing_status'].value_counts().rename_axis('routing_status').reset_index(name='routes'))

,input_variable,matched_role,rule,entry,assessment_grain,routing_status,missing_roles,missing_declarations
0,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,row_and_cell,READY_FOR_EVALUATION,,
1,CURR_DPD,arrears_measure,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_arrears,segment_and_period,READY_FOR_EVALUATION,,
2,INT_RT,priced_rate,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_tracking_rate,segment_and_period,READY_FOR_EVALUATION,,
3,BRR,internal_grade,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_internal_grade,segment_and_declared_review_cycle,UNSCOPED,,lower_frequency_fields_and_review_cycles
4,NOI,income_measure,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_income,segment_and_period,READY_FOR_EVALUATION,,
5,OCC_PCT,utilisation_measure,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_utilisation,segment_and_period,READY_FOR_EVALUATION,,
6,BAL_AMT,drawn_balance,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_ead_drawn_balance,entity_and_declared_update_cycle,UNSCOPED,exposure_change_indicators,behavioural_balance_update_frequency
7,CURR_DPD,arrears_measure,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_lgd_arrears,row_and_cell,UNSCOPED,non_arrears_trigger,
8,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,row_and_cell,READY_FOR_EVALUATION,,
9,SPREAD_BPS,spread_over_benchmark,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_fixed_rate_spread,row_and_cell,READY_FOR_EVALUATION,,


,routing_status,routes
0,READY_FOR_EVALUATION,9
1,UNSCOPED,3


## 5. Cell assessments, final tags, and source-value overlay

In [7]:
display(execution.assessment_ledger['status'].value_counts().rename_axis('assessment_status').reset_index(name='count'))
display(execution.raw_cell_claims.head(20))
resolved = execution.resolved_cell_tags
if resolved.empty:
    display(Markdown('**No final cell tags were generated.** Review bindings, declarations, and unscoped routes.'))
else:
    display(resolved['tag'].value_counts().rename_axis('final_tag').reset_index(name='cells'))
    display(resolved.head(20))
    symbols = {'CENSORED': '[RED]', 'STALE_FROZEN': '[AMBER]', 'NOT_APPLICABLE': '[BLUE]'}
    tag_sample = resolved.head(15)
    row_refs = tag_sample['row_reference'].drop_duplicates().tolist()
    overlay = data.set_index(row_reference_column).loc[row_refs].astype(object)
    for item in tag_sample.to_dict(orient='records'):
        value = overlay.at[item['row_reference'], item['input_variable']]
        overlay.at[item['row_reference'], item['input_variable']] = f"{value} {symbols[item['tag']]} {item['tag']}"
    display(overlay.reset_index())

,assessment_status,count
0,NO_TAG,7250
1,APPLIED_TAG,3550
2,UNSCOPED,3


,row_reference,input_variable,matched_role,rule,entry,tag,reason_code,input_value,is_populated
0,PD001_09,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
1,PD001_10,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
2,PD001_11,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
3,PD001_12,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
4,PD002_09,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
5,PD002_10,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
6,PD002_11,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
7,PD002_12,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
8,PD003_09,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False
9,PD003_10,DFLT_12M,forward_labels,t2_d08_valsim_rule_censored,t2_d08_valsim_censored_pd_incomplete_forward_window,CENSORED,incomplete_forward_window,NaN,False


,final_tag,cells
0,NOT_APPLICABLE,3096
1,CENSORED,400
2,STALE_FROZEN,54


,row_reference,input_variable,matched_role,rule,entry,tag,reason_code,input_value,is_populated,suppressed_tags,claim_count
0,PD001_01,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
1,PD001_01,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
2,PD001_02,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
3,PD001_02,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
4,PD001_03,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
5,PD001_03,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
6,PD001_04,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
7,PD001_04,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_before_maturity,NOT_APPLICABLE,before_contractual_maturity,0.00,True,,1
8,PD001_05,DEFAULT_FLAG,construction_constants,t2_d08_valsim_rule_not_applicable,t2_d08_valsim_not_applicable_pd_construction_constant,NOT_APPLICABLE,panel_scope_exclusion,0.00,True,,1
9,PD001_05,INT_RT,priced_rate,t2_d08_valsim_rule_stale_frozen,t2_d08_valsim_stale_frozen_tracking_rate,STALE_FROZEN,zero_variance_at_declared_grain,4.25,True,,1


,ROW_ID,FACILITY_ID,RPT_QTR,PORTF_SEG,CURR_DPD,INT_RT,RATE_TYPE,BRR,NOI,OCC_PCT,DFLT_12M,DEFAULT_FLAG,SPREAD_BPS,MAT_DT,MAT_BALLOON_IND,PMT_TYPE,PRIN_PAYDOWN_AMT,ORIG_LTV,BAL_AMT
0,PD001_01,F0001,2022-Q1,NORTH,0,3.53,FLOATING,G2,50130.0,0.69,0.0,0 [BLUE] NOT_APPLICABLE,176.0,2030-12-31,0 [BLUE] NOT_APPLICABLE,AMORTISING,1000,0.7,250000
1,PD001_02,F0001,2022-Q2,NORTH,0,3.61,FLOATING,G2,50370.0,0.691,0.0,0 [BLUE] NOT_APPLICABLE,176.0,2030-12-31,0 [BLUE] NOT_APPLICABLE,AMORTISING,1000,0.7,249000
2,PD001_03,F0001,2022-Q3,NORTH,0,3.69,FLOATING,G2,50610.0,0.692,0.0,0 [BLUE] NOT_APPLICABLE,176.0,2030-12-31,0 [BLUE] NOT_APPLICABLE,AMORTISING,1000,0.7,248000
3,PD001_04,F0001,2022-Q4,NORTH,0,3.77,FLOATING,G2,50850.0,0.693,0.0,0 [BLUE] NOT_APPLICABLE,176.0,2030-12-31,0 [BLUE] NOT_APPLICABLE,AMORTISING,1000,0.7,247000
4,PD001_05,F0001,2023-Q1,NORTH,0,4.25 [AMBER] STALE_FROZEN,FLOATING,G2,51090.0,0.694,0.0,0 [BLUE] NOT_APPLICABLE,176.0,2030-12-31,0 [BLUE] NOT_APPLICABLE,AMORTISING,1000,0.7,246000
5,PD001_06,F0001,2023-Q2,NORTH,0,4.25 [AMBER] STALE_FROZEN,FLOATING,G2,51330.0,0.695,0.0,0 [BLUE] NOT_APPLICABLE,176.0,2030-12-31,0 [BLUE] NOT_APPLICABLE,AMORTISING,1000,0.7,245000
6,PD001_07,F0001,2023-Q3,NORTH,0,4.25,FLOATING,G2,51570.0,0.696,0.0,0 [BLUE] NOT_APPLICABLE,176.0,2030-12-31,0,AMORTISING,1000,0.7,244000


## 6. Summarize the diagnostic outcome

In [8]:
for title, key in [
    ('Dataset summary', 'dataset_summary'), ('Field summary', 'field_summary'),
    ('Rule summary', 'rule_summary'), ('Group-grain coverage', 'group_summary'),
    ('Assessment outcomes', 'assessment_outcomes'), ('Unexamined fields', 'unexamined_fields'),
    ('Precedence resolutions', 'precedence_summary'),
    ('Populated values where field is not applicable', 'populated_where_not_applicable'),
]:
    display(Markdown(f'### {title}'))
    display(summaries[key].head(50))

### Dataset summary

,input_rows,input_columns,columns_with_roles,examined_fields,unexamined_fields,ready_rule_routes,unscoped_rule_routes,rule_assessments,distinct_tagged_cells,censored_cells,stale_frozen_cells,not_applicable_cells,unclassified_assessments,unscoped_entries,multi_claim_cells,populated_where_not_applicable,run_verdict
0,1200,19,18,9,10,9,3,10800,3550,400,54,3096,0,3,0,2700,PARTIALLY_CLASSIFIED


### Field summary

,input_variable,semantic_roles,total_rows,assessed_cells,tagged_cells,censored_count,stale_frozen_count,not_applicable_count,no_tag_cells,unclassified_assessments,unscoped_entries,tag_rate,examination_status
0,ROW_ID,,1200,0,0,0,0,0,0,0,0,NaN,NO_SEMANTIC_ROLE
1,FACILITY_ID,entity_id,1200,0,0,0,0,0,0,0,0,NaN,NO_RULE_ENTRY
2,RPT_QTR,period,1200,0,0,0,0,0,0,0,0,NaN,NO_RULE_ENTRY
3,PORTF_SEG,segment,1200,0,0,0,0,0,0,0,0,NaN,NO_RULE_ENTRY
4,CURR_DPD,arrears_measure,1200,1200,0,0,0,0,1200,0,1,0.000000,ASSESSED
5,INT_RT,priced_rate,1200,1200,51,0,51,0,1149,0,0,0.042500,ASSESSED
6,RATE_TYPE,rate_basis,1200,0,0,0,0,0,0,0,0,NaN,NO_RULE_ENTRY
7,BRR,internal_grade,1200,0,0,0,0,0,0,0,1,NaN,UNSCOPED
8,NOI,income_measure,1200,1200,3,0,3,0,1197,0,0,0.002500,ASSESSED
9,OCC_PCT,utilisation_measure,1200,1200,0,0,0,0,1200,0,0,0.000000,ASSESSED


### Rule summary

...

,input_variable,semantic_roles,examination_status,unscoped_entries
0,ROW_ID,,NO_SEMANTIC_ROLE,0
1,FACILITY_ID,entity_id,NO_RULE_ENTRY,0
2,RPT_QTR,period,NO_RULE_ENTRY,0
3,PORTF_SEG,segment,NO_RULE_ENTRY,0
4,RATE_TYPE,rate_basis,NO_RULE_ENTRY,0
5,BRR,internal_grade,UNSCOPED,1
6,MAT_DT,maturity_date,NO_RULE_ENTRY,0
7,PMT_TYPE,amortisation_basis,NO_RULE_ENTRY,0
8,ORIG_LTV,static_attributes,NO_RULE_ENTRY,0
9,BAL_AMT,drawn_balance,UNSCOPED,1


### Precedence resolutions

,row_reference,input_variable,claim_count,winning_tag,suppressed_tags


### Populated values where field is not applicable

,row_reference,input_variable,matched_role,entry,input_value
0,PD001_01,DEFAULT_FLAG,construction_constants,t2_d08_valsim_not_applicable_pd_construction_constant,0.0
1,PD001_01,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_not_applicable_pd_before_maturity,0.0
2,PD001_02,DEFAULT_FLAG,construction_constants,t2_d08_valsim_not_applicable_pd_construction_constant,0.0
3,PD001_02,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_not_applicable_pd_before_maturity,0.0
4,PD001_03,DEFAULT_FLAG,construction_constants,t2_d08_valsim_not_applicable_pd_construction_constant,0.0
5,PD001_03,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_not_applicable_pd_before_maturity,0.0
6,PD001_04,DEFAULT_FLAG,construction_constants,t2_d08_valsim_not_applicable_pd_construction_constant,0.0
7,PD001_04,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_not_applicable_pd_before_maturity,0.0
8,PD001_05,DEFAULT_FLAG,construction_constants,t2_d08_valsim_not_applicable_pd_construction_constant,0.0
9,PD001_05,MAT_BALLOON_IND,maturity_conditional_flags,t2_d08_valsim_not_applicable_pd_before_maturity,0.0


## 7. Fixture-only expectation comparison

This comparison is available only for synthetic fixtures. Older expectation files are partial raw claims, so an `ADDITIONAL_KB_FINDING` is not automatically an executor error.

In [9]:
expected_comparison = None
if INPUT_SOURCE == 'fixture':
    expected = fixture['legacy_expected_claims'].rename(columns={
        'row_id': 'row_reference', 'column_name': 'input_variable', 'expected_tag': 'tag',
    })
    keys = ['row_reference', 'input_variable', 'tag']
    expected_comparison = execution.raw_cell_claims[keys].drop_duplicates().merge(
        expected[keys].drop_duplicates(), on=keys, how='outer', indicator=True,
    )
    expected_comparison['comparison'] = expected_comparison.pop('_merge').map({
        'both': 'EXPECTED_AND_GENERATED', 'left_only': 'ADDITIONAL_KB_FINDING',
        'right_only': 'EXPECTED_NOT_GENERATED',
    }).astype(str)
    display(expected_comparison['comparison'].value_counts().rename_axis('comparison').reset_index(name='cells'))
    display(expected_comparison.loc[expected_comparison['comparison'] != 'EXPECTED_AND_GENERATED'].head(30))
else:
    print('Skipped: repository snapshots do not have fixture ground truth.')

,comparison,cells
0,EXPECTED_AND_GENERATED,1999
1,ADDITIONAL_KB_FINDING,1551


,row_reference,input_variable,tag,comparison
1,PD001_01,MAT_BALLOON_IND,NOT_APPLICABLE,ADDITIONAL_KB_FINDING
3,PD001_02,MAT_BALLOON_IND,NOT_APPLICABLE,ADDITIONAL_KB_FINDING
5,PD001_03,MAT_BALLOON_IND,NOT_APPLICABLE,ADDITIONAL_KB_FINDING
7,PD001_04,MAT_BALLOON_IND,NOT_APPLICABLE,ADDITIONAL_KB_FINDING
9,PD001_05,INT_RT,STALE_FROZEN,ADDITIONAL_KB_FINDING
10,PD001_05,MAT_BALLOON_IND,NOT_APPLICABLE,ADDITIONAL_KB_FINDING
12,PD001_06,INT_RT,STALE_FROZEN,ADDITIONAL_KB_FINDING
13,PD001_06,MAT_BALLOON_IND,NOT_APPLICABLE,ADDITIONAL_KB_FINDING
15,PD001_07,INT_RT,STALE_FROZEN,ADDITIONAL_KB_FINDING
16,PD001_07,MAT_BALLOON_IND,NOT_APPLICABLE,ADDITIONAL_KB_FINDING


## 8. Export this run

In [10]:
output_path = export_execution(
    source_label, execution, summaries, OUTPUT_ROOT, expected_comparison=expected_comparison,
    extra_frames={
        'column_binding_results': binding_results,
        **({'dictionary_evidence_comparison': comparison} if COMPARE_DICTIONARY_MODES else {}),
    },
    manifest_context={
        'input_source': INPUT_SOURCE, 'data_rows': len(data),
        'snapshot_id': SNAPSHOT_ID if INPUT_SOURCE == 'snapshot' else None,
        'table': TABLE_NAME if INPUT_SOURCE == 'snapshot' else None,
        'aar_sourced_dictionary_used': USE_AAR_SOURCED_DICTIONARY if INPUT_SOURCE == 'snapshot' else None,
        'dictionary_metadata_used_for_matching': USE_DICTIONARY_FOR_MATCHING,
        'external_dictionary_path': str(DICTIONARY_PATH) if DICTIONARY_PATH else None,
        'llm_fallback_enabled': USE_LLM_FALLBACK, 'binding_authority': binding_authority,
        'selected_column_count': len(selected_dictionary),
    },
)
print('Exported:', output_path)

Exported: C:\Src\DataWorkbench\experiments\test-lab\t2_d08_Value_semantics\output\cell_execution_v0_1\pd


## Interpretation

A reliable run has reviewed execution-ready bindings, the runtime declarations required by applicable rules, and an understood `UNSCOPED` population. Use the dictionary-on/off comparison to measure how much business metadata changes retrieval. Use LLM output as a proposed semantic binding subject to review; diagnostic flags themselves remain reproducible KB outcomes.